## Setup

In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
from collections import defaultdict
import json
import re
import unicodedata

vocabulary_occurrences = pd.read_csv("./QuranTotalMergedDatabase.csv")
qf_database = pd.read_csv("./QF_Database.csv")
database = pd.read_parquet("./qac_morphology.parquet")

key = ["Surah", "Ayah", "Word"]
QURANMORPH_PATH = "./quran-morph.csv"

DAGGER_ALIF = "\u0670"
QURANIC_MADDA = "\u0653"
WIKTIONARY_PATH = "./kaikki.org-dictionary-Arabic.jsonl"

vocabulary_occurrences

In [ ]:
frequency = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")
    .size()
    .sort_values(ascending=False)
    .rename("Frequency")
)

frequency_rank = (
    frequency
    .rank(method="first", ascending=False)
    .astype(int)
    .rename("Frequency Rank")
)

In [ ]:
assert (
    vocabulary_occurrences
    .groupby("Vocabulary ID")["Lemma"]
    .nunique()
    .le(1)
    .all()
)

In [ ]:
def unique_sorted(series):
    return sorted(series.dropna().unique().tolist())


def unique_in_order(series):
    return list(dict.fromkeys(series.dropna().tolist()))


def single_unique(series):
    values = series.dropna().unique()

    if len(values) == 0:
        return None

    if len(values) == 1:
        return values[0]

    return list(values)


# -----------------------
# Occurrence-level helpers
# -----------------------

vocabulary_occurrences["Is Participle"] = (
    vocabulary_occurrences["Morphology"]
    .str.contains(r"\bPCPL\b", regex=True, na=False)
)


def get_participle_type(morphology):
    if "ACT|PCPL" in morphology:
        return "Active"
    elif "PASS|PCPL" in morphology:
        return "Passive"
    return None


vocabulary_occurrences["Participle Type"] = (
    vocabulary_occurrences["Morphology"]
    .apply(get_participle_type)
)


# -----------------------
# Basic lexeme table
# -----------------------

vocabulary = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")
    .agg(
        Lemma=("Lemma", single_unique),
        Root=("Root", single_unique),
        POS=("POS", unique_sorted),
        IsParticiple=("Is Participle", "first"),
        ParticipleType=("Participle Type", "first"),
        Surahs=("Surah", unique_sorted),
        Juzs=("Juz", unique_sorted),
        ObservedForms=("Arabic", unique_in_order),
    )
)

print("After basic aggregation:", vocabulary.shape)


# -----------------------
# Frequency
# -----------------------

frequency.name = "Frequency"
frequency_rank.name = "Frequency Rank"

vocabulary = (
    vocabulary
    .join(frequency)
    .join(frequency_rank)
)

print("After frequency:", vocabulary.shape)


# -----------------------
# Counts of varying fields
# -----------------------

english_glosses = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")["English"]
    .agg(lambda x: Counter(x.dropna()))
    .rename("English Glosses")
)

vocabulary = vocabulary.join(english_glosses)

print("After English glosses:", vocabulary.shape)


form_counts = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")["Arabic"]
    .agg(lambda x: Counter(x.dropna()))
    .rename("Observed Form Counts")
)

vocabulary = vocabulary.join(form_counts)

print("After form counts:", vocabulary.shape)


pos_counts = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")["POS"]
    .agg(lambda x: Counter(x.dropna()))
    .rename("POS Counts")
)

vocabulary = vocabulary.join(pos_counts)

print("After POS counts:", vocabulary.shape)


# -----------------------
# Occurrence pointers
# -----------------------

occurrences = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")
    .apply(
        lambda g: list(
            zip(
                g["Surah"],
                g["Ayah"],
                g["Word"],
                g["Segment"]
            )
        ),
        include_groups=False
    )
    .rename("Occurrences")
)

vocabulary = vocabulary.join(occurrences)

print("After occurrences:", vocabulary.shape)


# Canonical deck order
vocabulary = vocabulary.sort_values("Frequency Rank")

assert len(vocabulary) == vocabulary_occurrences["Vocabulary ID"].nunique()

assert vocabulary["Frequency"].sum() == len(vocabulary_occurrences)

assert vocabulary["Frequency Rank"].is_unique

assert vocabulary["Frequency Rank"].min() == 1

assert vocabulary["Frequency Rank"].max() == len(vocabulary)

vocabulary

In [ ]:
word_structure = (
    database
    .groupby(key)
    .agg(
        QACSegmentCount=("Segment", "size"),
        PrefixCount=(
            "Morphology",
            lambda x: x.str.startswith("PREFIX").sum()
        ),
        SuffixCount=(
            "Morphology",
            lambda x: x.str.startswith("SUFFIX").sum()
        ),
    )
    .reset_index()
)

vocab_segment_count = (
    vocabulary_occurrences
    .groupby(key)
    .size()
    .rename("VocabularySegmentCount")
    .reset_index()
)

word_structure = word_structure.merge(
    vocab_segment_count,
    on=key,
    how="left",
    validate="one_to_one"
)

vocabulary_occurrences = vocabulary_occurrences.merge(
    word_structure,
    on=key,
    how="left",
    validate="many_to_one"
)

vocabulary_occurrences["Clean Occurrence"] = (
    (vocabulary_occurrences["QACSegmentCount"] == 1)
    & (vocabulary_occurrences["VocabularySegmentCount"] == 1)
)

clean_coverage = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")["Clean Occurrence"]
    .any()
)

clean_coverage.value_counts()

In [ ]:
no_clean_ids = clean_coverage[
    ~clean_coverage
].index

no_clean_occurrences = vocabulary_occurrences[
    vocabulary_occurrences["Vocabulary ID"].isin(no_clean_ids)
]

(
    no_clean_occurrences[
        ["PrefixCount", "SuffixCount", "VocabularySegmentCount"]
    ]
    .value_counts()
    .head(20)
)

vocabulary_occurrences["Simple Prefixed Occurrence"] = (
    (vocabulary_occurrences["VocabularySegmentCount"] == 1)
    & (vocabulary_occurrences["SuffixCount"] == 0)
)

usable_coverage = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")["Simple Prefixed Occurrence"]
    .any()
)

usable_coverage.value_counts()

In [ ]:
vocabulary_occurrences["Occurrence Tier"] = np.select(
    [
        vocabulary_occurrences["Clean Occurrence"],

        vocabulary_occurrences["Simple Prefixed Occurrence"],

        (
            (vocabulary_occurrences["VocabularySegmentCount"] == 1)
            & (vocabulary_occurrences["SuffixCount"] > 0)
        ),
    ],
    [
        1,  # stem alone
        2,  # prefix(es) + one stem, no suffix
        3,  # one stem + suffix(es)
    ],
    default=4  # multiple vocabulary-bearing stems
)

best_tier = (
    vocabulary_occurrences
    .groupby("Vocabulary ID")["Occurrence Tier"]
    .min()
    .rename("Best Occurrence Tier")
)

vocabulary = vocabulary.join(best_tier)

In [ ]:
vocabulary_occurrences["Best Occurrence Tier"] = (
    vocabulary_occurrences["Vocabulary ID"]
    .map(best_tier)
)

best_occurrences = vocabulary_occurrences[
    vocabulary_occurrences["Occurrence Tier"]
    ==
    vocabulary_occurrences["Best Occurrence Tier"]
].copy()

In [ ]:
# Identify useful segment types in the full QAC dataframe
database["IsPrefix"] = database["Morphology"].str.startswith("PREFIX")
database["IsSuffix"] = database["Morphology"].str.startswith("SUFFIX")

database["ConjunctionPrefix"] = (
    database["IsPrefix"] & database["POS"].eq("CONJ")
)

database["PrepositionPrefix"] = (
    database["IsPrefix"] & database["POS"].eq("P")
)

database["DeterminerPrefix"] = (
    database["IsPrefix"] & database["POS"].eq("DET")
)

database["PronounSuffix"] = (
    database["IsSuffix"] & database["POS"].eq("PRON")
)


word_features = (
    database
    .groupby(key)
    .agg(
        HasConjunctionPrefix=("ConjunctionPrefix", "any"),
        HasPrepositionPrefix=("PrepositionPrefix", "any"),
        HasDeterminerPrefix=("DeterminerPrefix", "any"),
        HasPronounSuffix=("PronounSuffix", "any"),
    )
    .reset_index()
)

In [ ]:
vocabulary_occurrences = vocabulary_occurrences.merge(
    word_features,
    on=key,
    how="left",
    validate="many_to_one"
)

In [ ]:
def clean_english_gloss(row):
    gloss = row["English"]

    if pd.isna(gloss):
        return None

    gloss = gloss.strip()

    # QF sometimes puts English-only grammatical material
    # in parentheses at the very beginning:
    # "(of) Allah" -> "Allah"
    # "(to) them" -> "them"
    gloss = re.sub(
        r"^\((?:of|to|in|on|at|by|with|from|for)\)\s*",
        "",
        gloss,
        flags=re.IGNORECASE
    )

    # Remove meaning supplied by an attached conjunction.
    if row["HasConjunctionPrefix"]:
        gloss = re.sub(
            r"^(?:and|then|so|but|or)\s+",
            "",
            gloss,
            flags=re.IGNORECASE
        )

    # Remove meaning supplied by an attached preposition.
    if row["HasPrepositionPrefix"]:
        gloss = re.sub(
            r"^(?:in|on|at|by|with|from|for|to|of)\s+",
            "",
            gloss,
            flags=re.IGNORECASE
        )

    # Remove English definite article when QAC actually
    # identifies an Arabic determiner prefix.
    if row["HasDeterminerPrefix"]:
        gloss = re.sub(
            r"^the\s+",
            "",
            gloss,
            flags=re.IGNORECASE
        )

    # Possessive/object pronouns contributed by suffixes.
    if row["HasPronounSuffix"]:
        # "their inscription" -> "inscription"
        gloss = re.sub(
            r"^(?:my|your|his|her|its|our|their)\s+",
            "",
            gloss,
            flags=re.IGNORECASE
        )

        # "created them" -> "created"
        gloss = re.sub(
            r"\s+(?:me|us|you|him|her|it|them)$",
            "",
            gloss,
            flags=re.IGNORECASE
        )

    # Whitespace cleanup
    gloss = re.sub(r"\s+", " ", gloss).strip()

    return gloss

In [ ]:
vocabulary_occurrences["Cleaned English"] = (
    vocabulary_occurrences.apply(
        clean_english_gloss,
        axis=1
    )
)


# Rebuild this subset now that Cleaned English exists.
best_occurrences = vocabulary_occurrences[
    vocabulary_occurrences["Occurrence Tier"]
    ==
    vocabulary_occurrences["Best Occurrence Tier"]
].copy()


definition_evidence = (
    best_occurrences
    .groupby("Vocabulary ID")["Cleaned English"]
    .agg(lambda x: Counter(x.dropna()))
    .rename("Definition Evidence")
)

vocabulary = vocabulary.join(
    definition_evidence
)

In [ ]:
vocabulary_occurrences[
    vocabulary_occurrences["English"]
    != vocabulary_occurrences["Cleaned English"]
][
    [
        "Lemma",
        "Arabic",
        "English",
        "Cleaned English",
        "HasConjunctionPrefix",
        "HasPrepositionPrefix",
        "HasDeterminerPrefix",
        "HasPronounSuffix",
    ]
].sample(50)

## QuranMorph

In [ ]:
quranmorph = pd.read_csv(
    QURANMORPH_PATH,
    keep_default_na=False,
)

quranmorph = quranmorph.rename(
    columns={
        "surah_number": "Surah",
        "verse_number": "Ayah",
        "word_position": "Word",
        "word": "QuranMorph Word",
        "POS": "QuranMorph POS",
        "qabas_lemma_id": "QuranMorph Lemma ID",
        "qabas_lemma": "QuranMorph Raw Lemma",
    }
)


# ------------------------------------------------------------
# Parse QuranMorph / Qabas lemmas
# ------------------------------------------------------------

def parse_quranmorph_lemma(raw_lemma):
    """
    Parse Qabas/QuranMorph lemma alternatives.

    Examples:
        جِنَّةٌ2
            -> ["جِنَّةٌ"]

        ثَمُودُ | ثَمُودٌ
            -> ["ثَمُودُ", "ثَمُودٌ"]

        جَحَدَ | جَحَدَ ب
            -> ["جَحَدَ", "جَحَدَ ب"]

    Trailing numerical homograph identifiers are removed.
    The original raw field is preserved separately.
    """
    if not isinstance(raw_lemma, str) or not raw_lemma:
        return []

    alternatives = []

    for lemma in raw_lemma.split("|"):
        lemma = lemma.strip()

        lemma = re.sub(
            r"\d+$",
            "",
            lemma,
        ).strip()

        if lemma:
            alternatives.append(lemma)

    return list(dict.fromkeys(alternatives))


def quranmorph_match_forms(alternatives):
    """
    Convert QuranMorph/Qabas lemma alternatives into forms
    suitable for dictionary lookup.

    Qabas may append valency information:
        جَحَدَ ب

    For dictionary lookup, use:
        جَحَدَ

    The complete Qabas form remains preserved separately.
    """
    if not isinstance(alternatives, list):
        return []

    forms = []

    for lemma in alternatives:
        lemma = lemma.strip()

        if not lemma:
            continue

        lexical_form = lemma.split()[0]

        if lexical_form:
            forms.append(lexical_form)

    return list(dict.fromkeys(forms))


quranmorph["QuranMorph Lemma Alternatives"] = (
    quranmorph["QuranMorph Raw Lemma"]
    .apply(parse_quranmorph_lemma)
)

quranmorph["QuranMorph Match Forms"] = (
    quranmorph["QuranMorph Lemma Alternatives"]
    .apply(quranmorph_match_forms)
)


# ------------------------------------------------------------
# Verify alignment with Quran Foundation
# ------------------------------------------------------------

print(
    "QuranMorph rows:",
    len(quranmorph)
)

print(
    "Duplicate QuranMorph word keys:",
    quranmorph.duplicated(key).sum()
)


qf_words = qf_database.copy()

if "char_type_name" in qf_words.columns:
    qf_words = qf_words[
        qf_words["char_type_name"] == "word"
    ]


qf_keys = set(
    map(
        tuple,
        qf_words[key]
        .drop_duplicates()
        .to_numpy()
    )
)

quranmorph_keys = set(
    map(
        tuple,
        quranmorph[key]
        .drop_duplicates()
        .to_numpy()
    )
)

print(
    "QF keys missing from QuranMorph:",
    len(qf_keys - quranmorph_keys)
)

print(
    "QuranMorph keys missing from QF:",
    len(quranmorph_keys - qf_keys)
)


# ------------------------------------------------------------
# Map QuranMorph words onto QAC vocabulary occurrences
# ------------------------------------------------------------

quranmorph_lookup = quranmorph[
    key
    + [
        "QuranMorph Word",
        "QuranMorph POS",
        "QuranMorph Lemma ID",
        "QuranMorph Raw Lemma",
        "QuranMorph Lemma Alternatives",
        "QuranMorph Match Forms",
    ]
].copy()


quranmorph_occurrence_map = (
    vocabulary_occurrences[
        key
        + [
            "Vocabulary ID",
            "VocabularySegmentCount",
        ]
    ]
    .merge(
        quranmorph_lookup,
        on=key,
        how="left",
        validate="many_to_one",
    )
)


# Only use QuranMorph automatically when the orthographic word
# contains exactly one QAC vocabulary-bearing segment.
quranmorph_clean_map = (
    quranmorph_occurrence_map[
        quranmorph_occurrence_map[
            "VocabularySegmentCount"
        ] == 1
    ]
    .copy()
)


# ------------------------------------------------------------
# Aggregate QuranMorph evidence by QAC Vocabulary ID
# ------------------------------------------------------------

def unique_nonempty_in_order(series):
    """
    Collect unique scalar values while preserving first-seen order.
    """
    return list(
        dict.fromkeys(
            value
            for value in series
            if value is not None
            and value != ""
        )
    )


def flatten_unique_in_order(series):
    """
    Flatten list-valued cells and collect unique values while
    preserving first-seen order.
    """
    result = []
    seen = set()

    for value in series:

        if isinstance(value, list):
            values = value

        elif value is None or value == "":
            continue

        else:
            values = [value]

        for item in values:
            if item not in seen:
                seen.add(item)
                result.append(item)

    return result


quranmorph_by_vocab = (
    quranmorph_clean_map
    .groupby("Vocabulary ID")
    .agg(
        QuranMorphRawLemmas=(
            "QuranMorph Raw Lemma",
            unique_nonempty_in_order,
        ),
        QuranMorphLemmaAlternatives=(
            "QuranMorph Lemma Alternatives",
            flatten_unique_in_order,
        ),
        QuranMorphMatchForms=(
            "QuranMorph Match Forms",
            flatten_unique_in_order,
        ),
        QuranMorphLemmaIDs=(
            "QuranMorph Lemma ID",
            unique_nonempty_in_order,
        ),
        QuranMorphPOS=(
            "QuranMorph POS",
            unique_nonempty_in_order,
        ),
    )
)


# ------------------------------------------------------------
# Join QuranMorph evidence onto vocabulary
# ------------------------------------------------------------

columns_to_replace = (
    quranmorph_by_vocab.columns
    .intersection(vocabulary.columns)
)

vocabulary = (
    vocabulary
    .drop(columns=columns_to_replace)
    .join(quranmorph_by_vocab)
)


quranmorph_list_columns = [
    "QuranMorphRawLemmas",
    "QuranMorphLemmaAlternatives",
    "QuranMorphMatchForms",
    "QuranMorphLemmaIDs",
    "QuranMorphPOS",
]

for column in quranmorph_list_columns:
    vocabulary[column] = (
        vocabulary[column]
        .apply(
            lambda value:
                value
                if isinstance(value, list)
                else []
        )
    )


vocabulary["QuranMorph Match Form Count"] = (
    vocabulary["QuranMorphMatchForms"]
    .apply(len)
)

## Wiktionary Normalization

In [ ]:
def _arabic_match_key_core(
    text,
    expand_dagger_alif=False
):
    if not isinstance(text, str) or not text:
        return None

    # IMPORTANT:
    # Handle Quranic seatless hamza BEFORE Unicode normalization.
    # NFC can reorder adjacent combining marks and prevent these
    # sequences from matching afterward.
    text = (
        text
        .replace("ـٔ", "ء")
        .replace("ـٕ", "ء")
    )

    text = unicodedata.normalize("NFC", text)

    # Normalize ordinary hamza carriers while preserving hamza.
    text = (
        text
        .replace("آ", "ءا")
        .replace("أ", "ء")
        .replace("إ", "ء")
        .replace("ؤ", "ء")
        .replace("ئ", "ء")
    )

    # Alif wasla
    text = text.replace("ٱ", "ا")

    if expand_dagger_alif:
        text = text.replace(DAGGER_ALIF, "ا")

    # Remove remaining tatweel
    text = text.replace("ـ", "")

    # Remove vowels and remaining Quranic combining marks
    text = "".join(
        char for char in text
        if unicodedata.category(char) != "Mn"
    )

    return text

def qac_match_key(
    text,
    expand_dagger_alif=False
):
    if not isinstance(text, str) or not text:
        return None

    # IMPORTANT:
    # Remove Quranic recitation madda BEFORE NFC can
    # compose ا + ٓ into lexical-looking آ.
    text = text.replace(QURANIC_MADDA, "")

    return _arabic_match_key_core(
        text,
        expand_dagger_alif=expand_dagger_alif
    )

def wiktionary_match_key(text):
    return _arabic_match_key_core(text)

def qac_match_key_relaxed(text):
    key = qac_match_key(text)

    if key is None:
        return None

    return key.replace("ى", "ي")


def wiktionary_match_key_relaxed(text):
    key = wiktionary_match_key(text)

    if key is None:
        return None

    return key.replace("ى", "ي")

## Wiktionary Helpers

In [ ]:
def contains_arabic(text):
    return (
        isinstance(text, str)
        and any("\u0600" <= char <= "\u06FF" for char in text)
    )


def get_wiktionary_forms(entry):
    seen = set()

    word = entry.get("word")

    if contains_arabic(word):
        seen.add(("word", word))
        yield "word", word

    for form_data in entry.get("forms", []):
        form = form_data.get("form")
        candidate = ("form", form)

        if contains_arabic(form) and candidate not in seen:
            seen.add(candidate)
            yield "form", form

## POS Compatibility

In [ ]:
QAC_TO_WIKTIONARY_POS = {
    "V": {"verb"},

    # QAC N can include substantivized adjectives and numerals.
    "N": {"noun", "adj", "num"},

    # Adjectives can likewise be used substantivally.
    "ADJ": {"adj", "noun"},

    "PN": {"name", "noun"},
    "PRON": {"pron"},
    "DEM": {"pron", "det"},
    "REL": {"pron"},
    "P": {"prep"},
    "CONJ": {"conj"},
}


def allowed_wiktionary_pos(row):
    qac_pos = row["POS"]

    if not isinstance(qac_pos, (list, tuple, set)):
        return None

    allowed = set()
    has_unmapped = False

    for pos in qac_pos:
        if pos in QAC_TO_WIKTIONARY_POS:
            allowed.update(QAC_TO_WIKTIONARY_POS[pos])
        else:
            has_unmapped = True

    if row["IsParticiple"]:
        allowed.update({"noun", "adj"})

    if has_unmapped and not row["IsParticiple"]:
        return None

    return allowed or None

## Matching Strategies

In [ ]:
def lexical_match_queries(row):
    queries = [
        (
            "lemma-strict",
            row["Lemma"],
            qac_match_key(
                row["Lemma"]
            ),
            "standard",
        ),
        (
            "lemma-dagger",
            row["Lemma"],
            qac_match_key(
                row["Lemma"],
                expand_dagger_alif=True,
            ),
            "standard",
        ),
        (
            "lemma-relaxed",
            row["Lemma"],
            qac_match_key_relaxed(
                row["Lemma"]
            ),
            "relaxed",
        ),
    ]

    # QuranMorph/Qabas supplies independently curated
    # dictionary-lemma candidates.
    #
    # These use Wiktionary-style normalization because
    # QuranMorph is already in conventional Arabic
    # orthography rather than QAC Uthmani orthography.
    for form in row.get(
        "QuranMorphMatchForms",
        [],
    ):
        queries.append(
            (
                "quranmorph-strict",
                form,
                wiktionary_match_key(form),
                "standard",
            )
        )

        queries.append(
            (
                "quranmorph-relaxed",
                form,
                wiktionary_match_key_relaxed(form),
                "relaxed",
            )
        )

    for form in row["ObservedForms"]:
        queries.append(
            (
                "observed-strict",
                form,
                qac_match_key(form),
                "standard",
            )
        )

    seen = set()
    result = []

    for query in queries:
        signature = (
            query[2],
            query[3],
        )

        if (
            query[2] is not None
            and signature not in seen
        ):
            seen.add(signature)
            result.append(query)

    return result

## Kaikki

In [ ]:
# Needed keys

standard_target_keys = set()
relaxed_target_keys = set()

for _, row in vocabulary.iterrows():
    for _, _, key, index_type in lexical_match_queries(row):

        if index_type == "standard":
            standard_target_keys.add(key)

        elif index_type == "relaxed":
            relaxed_target_keys.add(key)

# Scan Kaiki

def build_wiktionary_indexes(
    path,
    standard_target_keys,
    relaxed_target_keys,
):
    standard_index = defaultdict(list)
    relaxed_index = defaultdict(list)

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)

            for source, form in get_wiktionary_forms(entry):

                # Standard Wiktionary normalization
                standard_key = wiktionary_match_key(form)

                if standard_key in standard_target_keys:
                    standard_index[standard_key].append({
                        "source": source,
                        "matched_form": form,
                        "entry": entry,
                    })

                # Relaxed Wiktionary normalization
                relaxed_key = wiktionary_match_key_relaxed(form)

                if relaxed_key in relaxed_target_keys:
                    relaxed_index[relaxed_key].append({
                        "source": source,
                        "matched_form": form,
                        "entry": entry,
                    })

    return dict(standard_index), dict(relaxed_index)

wiktionary_standard_index, wiktionary_relaxed_index = (
    build_wiktionary_indexes(
        WIKTIONARY_PATH,
        standard_target_keys,
        relaxed_target_keys,
    )
)

## Match Wiktionary

In [ ]:
def filter_by_pos(candidates, allowed_pos):
    if allowed_pos is None:
        return candidates

    return [
        candidate
        for candidate in candidates
        if candidate["entry"].get("pos") in allowed_pos
    ]

def match_wiktionary(row):
    allowed_pos = allowed_wiktionary_pos(row)

    best_raw_match = None

    for basis, query_form, key, index_type in lexical_match_queries(row):

        index = (
            wiktionary_standard_index
            if index_type == "standard"
            else wiktionary_relaxed_index
        )

        candidates = index.get(key, [])

        if not candidates:
            continue

        # Remember the first spelling match for diagnostics
        if best_raw_match is None:
            best_raw_match = {
                "basis": basis,
                "query_form": query_form,
                "key": key,
                "candidates": candidates,
            }

        compatible = filter_by_pos(
            candidates,
            allowed_pos
        )

        # This strategy produced a lexically plausible match
        if compatible:
            return {
                "Wiktionary Match Status": "matched",
                "Wiktionary Match Basis": basis,
                "Wiktionary Query Form": query_form,
                "Wiktionary Matched Key": key,
                "Wiktionary Candidates": candidates,
                "Wiktionary POS Candidates": compatible,
            }

    # We found the spelling but no compatible POS
    if best_raw_match is not None:
        return {
            "Wiktionary Match Status": "pos-conflict",
            "Wiktionary Match Basis": best_raw_match["basis"],
            "Wiktionary Query Form": best_raw_match["query_form"],
            "Wiktionary Matched Key": best_raw_match["key"],
            "Wiktionary Candidates": best_raw_match["candidates"],
            "Wiktionary POS Candidates": [],
        }

    # Nothing at all
    return {
        "Wiktionary Match Status": "unmatched",
        "Wiktionary Match Basis": None,
        "Wiktionary Query Form": None,
        "Wiktionary Matched Key": None,
        "Wiktionary Candidates": [],
        "Wiktionary POS Candidates": [],
    }

wiktionary_results = vocabulary.apply(
    match_wiktionary,
    axis=1,
    result_type="expand"
)

columns_to_replace = (
    wiktionary_results.columns
    .intersection(vocabulary.columns)
)

vocabulary = (
    vocabulary
    .drop(columns=columns_to_replace)
    .join(wiktionary_results)
)

## Wiktionary Headword Candidates

In [ ]:
# Preserve exactly what QAC supplied.
vocabulary["QAC Lemma"] = vocabulary["Lemma"]

def canonical_arabic_form(entry):
    """
    Return Wiktionary's vocalized canonical form when available.
    Fall back to the page word.
    """
    for form_data in entry.get("forms", []):
        form = form_data.get("form")
        tags = set(form_data.get("tags", []))

        if "canonical" in tags and contains_arabic(form):
            return form

    word = entry.get("word")

    if contains_arabic(word):
        return word

    return None


def unique_candidate_headword(candidates):
    """
    If all POS-compatible Wiktionary candidates agree on one
    canonical Arabic headword, return it. Otherwise return None.
    """
    if not isinstance(candidates, list) or not candidates:
        return None

    headwords = {
        headword
        for candidate in candidates
        if (
            headword := canonical_arabic_form(
                candidate["entry"]
            )
        ) is not None
    }

    if len(headwords) == 1:
        return next(iter(headwords))

    return None


vocabulary["Wiktionary Headword Candidate"] = (
    vocabulary["Wiktionary POS Candidates"]
    .apply(unique_candidate_headword)
)

## Diagnostics

In [ ]:
print("WIKTIONARY STATUS")
print(
    vocabulary[
        "Wiktionary Match Status"
    ].value_counts()
)

print()


matched = (
    vocabulary["Wiktionary Match Status"]
    == "matched"
)

occurrence_coverage = (
    vocabulary.loc[
        matched,
        "Frequency"
    ].sum()
    / vocabulary["Frequency"].sum()
)

print(
    f"Lexeme coverage: {matched.mean():.1%}"
)

print(
    f"Occurrence coverage: {occurrence_coverage:.1%}"
)

print(
    "Unique Wiktionary headword candidates:",
    vocabulary[
        "Wiktionary Headword Candidate"
    ].notna().sum()
)

print(
    "Manual-review lexemes:",
    (
        vocabulary["Wiktionary Match Status"]
        == "unmatched"
    ).sum()
)


# Final integrity checks
assert vocabulary.index.is_unique

assert (
    vocabulary["Frequency"].sum()
    == len(vocabulary_occurrences)
)

assert (
    vocabulary["Frequency Rank"].is_unique
)

assert (
    vocabulary["Frequency Rank"].min()
    == 1
)

assert (
    vocabulary["Frequency Rank"].max()
    == len(vocabulary)
)

In [ ]:
matched = (
    vocabulary["Wiktionary Match Status"] == "matched"
)

coverage = (
    vocabulary.loc[matched, "Frequency"].sum()
    / vocabulary["Frequency"].sum()
)

print(f"Lexeme coverage: {matched.mean():.1%}")
print(f"Occurrence coverage: {coverage:.1%}")

In [ ]:
remaining_unmatched = vocabulary[
    vocabulary["Wiktionary Match Status"]
    == "unmatched"
].copy()

remaining_unmatched[
    [
        "QAC Lemma",
        "QuranMorphMatchForms",
        "QuranMorphPOS",
        "POS",
        "Root",
        "Frequency",
        "English Glosses",
    ]
].sort_values(
    "Frequency",
    ascending=False,
).head(50)

## Manual Review Enrichment — CAMeL Morphology

In [ ]:
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

camel_db = MorphologyDB.builtin_db(
    "calima-msa-r13"
)

camel_analyzer = Analyzer(
    camel_db
)

STANDARD_ARABIC_DIACRITICS = {
    "\u064b",  # fathatan
    "\u064c",  # dammatan
    "\u064d",  # kasratan
    "\u064e",  # fatha
    "\u064f",  # damma
    "\u0650",  # kasra
    "\u0651",  # shadda
    "\u0652",  # sukun
}


def remove_initial_shadda(text):
    """
    Remove a shadda attached to the first Arabic base letter.

    In QAC/Uthmani text this can represent cross-word recitational
    assimilation, e.g. مُّقَرَّبُون, rather than lexical gemination.

    Internal shaddas are preserved.
    """
    result = []
    seen_first_letter = False
    seen_second_letter = False

    for char in text:
        category = unicodedata.category(char)

        if category.startswith("L"):
            if not seen_first_letter:
                seen_first_letter = True
            else:
                seen_second_letter = True

            result.append(char)
            continue

        if (
            char == "\u0651"
            and seen_first_letter
            and not seen_second_letter
        ):
            continue

        result.append(char)

    return "".join(result)


def qac_to_camel_form(
    text,
    expand_dagger_alif=True,
    keep_diacritics=True,
):
    """
    Convert QAC/Uthmani spelling into a form suitable for CAMeL.

    Unlike Wiktionary matching, this preserves ordinary Arabic
    vowel information because CAMeL can use it to disambiguate
    morphological analyses.
    """
    if not isinstance(text, str) or not text:
        return None

    # Qur'anic recitation madda is not lexical.
    text = text.replace(QURANIC_MADDA, "")

    # Convert Quranic seatless hamza before Unicode normalization.
    text = (
        text
        .replace("ـٔ", "ء")
        .replace("ـٕ", "ء")
    )

    text = unicodedata.normalize("NFC", text)

    # Alif wasla -> ordinary alif.
    text = text.replace("ٱ", "ا")

    # Convert dagger alif to ordinary alif for modern morphology.
    if expand_dagger_alif:
        text = text.replace(
            DAGGER_ALIF,
            "ا"
        )

    # Remove tatweel.
    text = text.replace("ـ", "")

    # Remove initial recitational assimilation shadda,
    # while retaining lexical/internal shadda.
    text = remove_initial_shadda(text)

    if keep_diacritics:
        # Keep ordinary Arabic vowels/shadda/sukun,
        # discard other Quranic combining marks.
        text = "".join(
            char
            for char in text
            if (
                unicodedata.category(char) != "Mn"
                or char in STANDARD_ARABIC_DIACRITICS
            )
        )

    else:
        # Unvocalized fallback.
        text = "".join(
            char
            for char in text
            if unicodedata.category(char) != "Mn"
        )

    return text

def camel_input_variants(row):
    """
    Produce CAMeL inputs in two confidence tiers:

    1. cleaned but vocalized QAC forms
    2. unvocalized forms as a fallback

    The analyzer should prefer tier 1 whenever possible.
    """
    source_forms = [
        ("lemma", row["QAC Lemma"]),
    ]

    for form in row["ObservedForms"]:
        source_forms.append(
            ("observed", form)
        )

    variants = []
    seen = set()

    for source, form in source_forms:
        if not isinstance(form, str) or not form:
            continue

        vocalized = qac_to_camel_form(
            form,
            keep_diacritics=True,
        )

        unvocalized = qac_to_camel_form(
            form,
            keep_diacritics=False,
        )

        for tier, candidate in [
            ("vocalized", vocalized),
            ("unvocalized", unvocalized),
        ]:
            signature = (
                tier,
                candidate,
            )

            if (
                candidate
                and signature not in seen
            ):
                seen.add(signature)

                variants.append({
                    "source": source,
                    "tier": tier,
                    "form": candidate,
                })

    return variants

def analyze_with_camel(row):
    queries = camel_input_variants(row)

    all_analyses = []
    successful_inputs = []

    analysis_tier = None

    # Try precise, vocalized forms first.
    for tier in [
        "vocalized",
        "unvocalized",
    ]:
        tier_results = []

        for query in queries:
            if query["tier"] != tier:
                continue

            analyses = camel_analyzer.analyze(
                query["form"]
            )

            if not analyses:
                continue

            successful_inputs.append(query)

            for analysis in analyses:
                tier_results.append({
                    "input_source": query["source"],
                    "input_tier": tier,
                    "input_form": query["form"],
                    "analysis": analysis,
                })

        # If CAMeL understood any vocalized form,
        # do NOT pollute it with ambiguous unvocalized analyses.
        if tier_results:
            all_analyses = tier_results
            analysis_tier = tier
            break

    # Deduplicate lexical analyses.
    unique = []
    seen = set()

    for result in all_analyses:
        analysis = result["analysis"]

        signature = (
            analysis.get("lex"),
            analysis.get("pos"),
            analysis.get("gloss"),
            analysis.get("num"),
            analysis.get("asp"),
            analysis.get("per"),
            analysis.get("vox"),
        )

        if signature not in seen:
            seen.add(signature)
            unique.append(result)

    headwords = sorted({
        result["analysis"].get("lex")
        for result in unique
        if result["analysis"].get("lex")
    })

    return {
        "CAMeL Analysis Tier": analysis_tier,
        "CAMeL Analysis Count": len(unique),
        "CAMeL Headwords": headwords,
        "CAMeL Headword Count": len(headwords),
        "CAMeL Analyses": unique,
        "CAMeL Successful Inputs": successful_inputs,
    }

camel_review = vocabulary[
    vocabulary["Wiktionary Match Status"]
    == "unmatched"
].copy()

camel_results = camel_review.apply(
    analyze_with_camel,
    axis=1,
    result_type="expand",
)

camel_review = camel_review.join(
    camel_results
)

print(
    camel_review[
        "CAMeL Headword Count"
    ].value_counts().sort_index()
)

has_camel_analysis = (
    camel_review["CAMeL Analysis Count"]
    > 0
)

print(
    "Any CAMeL analysis:",
    f"{has_camel_analysis.mean():.1%}",
)

unique_headword = (
    camel_review["CAMeL Headword Count"]
    == 1
)

print(
    "Exactly one CAMeL headword:",
    f"{unique_headword.mean():.1%}",
)

camel_review[
    [
        "QAC Lemma",
        "POS",
        "Root",
        "Frequency",
        "English Glosses",
        "CAMeL Headwords",
        "CAMeL Headword Count",
    ]
].sort_values(
    "Frequency",
    ascending=False
).head(50)

## Export Frozen Data

In [ ]:
def make_json_safe(value):
    """
    Recursively convert nested Python/numpy objects into values
    that json.dumps() can preserve reliably.
    """
    if isinstance(value, Counter):
        value = dict(value)

    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, set):
        return [
            make_json_safe(item)
            for item in sorted(
                value,
                key=str,
            )
        ]

    if isinstance(value, np.generic):
        return value.item()

    return value


def serialize_nested_value(value):
    """
    Serialize nested columns as valid JSON while leaving ordinary
    scalar CSV values untouched.
    """
    if isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            Counter,
        )
    ):
        return json.dumps(
            make_json_safe(value),
            ensure_ascii=False,
        )

    return value


def serialize_dataframe_for_csv(df):
    """
    Return a CSV-safe copy without modifying the production dataframe.
    """
    result = df.copy()

    for column in result.columns:
        result[column] = (
            result[column]
            .apply(serialize_nested_value)
        )

    return result

# Add only the compact CAMeL evidence that may be useful later.
camel_export_columns = [
    "CAMeL Analysis Tier",
    "CAMeL Analysis Count",
    "CAMeL Headwords",
    "CAMeL Headword Count",
]

frozen_vocabulary = vocabulary.copy()

frozen_vocabulary = frozen_vocabulary.join(
    camel_review[
        camel_export_columns
    ]
)


vocabulary_csv = (
    serialize_dataframe_for_csv(
        frozen_vocabulary
    )
    .reset_index()
)


vocabulary_csv.to_csv(
    "vocabulary_frozen.csv.gz",
    index=False,
    encoding="utf-8",
    compression="gzip",
)

print(
    "Frozen vocabulary rows:",
    len(vocabulary_csv)
)

vocabulary_occurrences_csv = (
    serialize_dataframe_for_csv(
        vocabulary_occurrences
    )
)


vocabulary_occurrences_csv.to_csv(
    "vocabulary_occurrences_frozen.csv.gz",
    index=False,
    encoding="utf-8",
    compression="gzip",
)

print(
    "Frozen vocabulary occurrences:",
    len(vocabulary_occurrences_csv)
)

manual_review_columns = [
    "QAC Lemma",
    "QuranMorphLemmaAlternatives",
    "QuranMorphMatchForms",
    "QuranMorphPOS",
    "POS",
    "Root",
    "IsParticiple",
    "ParticipleType",
    "Frequency",
    "ObservedForms",
    "English Glosses",
    "Definition Evidence",
]


manual_review = (
    vocabulary[
        vocabulary["Wiktionary Match Status"]
        == "unmatched"
    ][
        manual_review_columns
    ]
    .copy()
)


# Add compact CAMeL evidence.
manual_review = manual_review.join(
    camel_review[
        [
            "CAMeL Analysis Tier",
            "CAMeL Headwords",
            "CAMeL Headword Count",
        ]
    ]
)


# Fields you will actually edit by hand.
manual_review["Manual Headword"] = ""
manual_review["Manual Plural"] = ""
manual_review["Manual Definition"] = ""
manual_review["Manual Notes"] = ""
manual_review["Reviewed"] = False


manual_review = (
    manual_review
    .sort_values(
        "Frequency",
        ascending=False,
    )
    .reset_index()
)


for column in [
    "QuranMorphLemmaAlternatives",
    "QuranMorphMatchForms",
    "QuranMorphPOS",
    "POS",
    "ObservedForms",
    "English Glosses",
    "Definition Evidence",
    "CAMeL Headwords",
]:
    manual_review[column] = (
        manual_review[column]
        .apply(serialize_nested_value)
    )


assert manual_review["Vocabulary ID"].is_unique

assert len(manual_review) == (
    vocabulary["Wiktionary Match Status"]
    == "unmatched"
).sum()


manual_review.to_csv(
    "manual_lexeme_review.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    "Manual-review rows:",
    len(manual_review)
)

frozen_vocabulary.to_pickle(
    "vocabulary_frozen.pkl"
)

vocabulary_occurrences.to_pickle(
    "vocabulary_occurrences_frozen.pkl"
)